In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
excel_path = "/home/ubuntu/giodir/digitalPathology/data/aiFlopp/df_gg_reggio_aiforia.csv"

In [3]:
# open excel sheet in pandas

df = pd.read_csv(excel_path)

In [4]:
df.head()

,CODICE CASO,REPERE/VETRINO,GLEASON Principale_re,GLEASON Secondario_re,GG ISUP PER SINGOLO REPERE/VETRINO_re,GLEASON Principale_aiforia,GLEASON Secondario_aiforia,GG ISUP PER SINGOLO REPERE/VETRINO_aiforia
0,I-25-16291,A,3.0,3.0,1.0,3.0,3.0,1.0
1,I-25-16291,B,3.0,3.0,1.0,3.0,3.0,1.0
2,I-25-16291,C,NaN,NaN,NaN,4.0,4.0,4.0
3,I-25-16291,D,4.0,3.0,3.0,4.0,3.0,3.0
4,I-25-16291,E,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.columns.to_list()

['CODICE CASO',
 'REPERE/VETRINO',
 'GLEASON Principale_re',
 'GLEASON Secondario_re',
 'GG ISUP PER SINGOLO REPERE/VETRINO_re',
 'GLEASON Principale_aiforia',
 'GLEASON Secondario_aiforia',
 'GG ISUP PER SINGOLO REPERE/VETRINO_aiforia']

In [6]:
columns_to_keep = [
  'CODICE CASO',
 'REPERE/VETRINO',
 'GG ISUP PER SINGOLO REPERE/VETRINO_re',
 'GG ISUP PER SINGOLO REPERE/VETRINO_aiforia'
 ]

columns_new_names = {
  'CODICE CASO': "case_code",
 'REPERE/VETRINO': "bersaglio",
 'GG ISUP PER SINGOLO REPERE/VETRINO_re': "GG_reggio",
 'GG ISUP PER SINGOLO REPERE/VETRINO_aiforia': "GG_aiforia"
 }


filtered_df = df[columns_to_keep]
filtered_df.columns = [columns_new_names.get(col, col) for col in filtered_df.columns]

In [7]:
filtered_df.head()

,case_code,bersaglio,GG_reggio,GG_aiforia
0,I-25-16291,A,1.0,1.0
1,I-25-16291,B,1.0,1.0
2,I-25-16291,C,NaN,4.0
3,I-25-16291,D,3.0,3.0
4,I-25-16291,E,NaN,NaN


In [8]:
# Assign -1 values to case where GG is missing (meaning there is no tumor)
filtered_df['GG_reggio'] = filtered_df['GG_reggio'].fillna(-1)
filtered_df['GG_aiforia'] = filtered_df['GG_aiforia'].fillna(-1)

In [9]:
filtered_df["difference"] = np.abs(filtered_df['GG_reggio'] - filtered_df['GG_aiforia'])

In [10]:
filtered_df["difference"].value_counts()

difference
0.0    400
1.0     66
2.0     40
5.0     18
3.0      8
4.0      3
6.0      3
Name: count, dtype: int64

In [11]:
len(filtered_df)

538

In [12]:
# Parse labels to get the bag_id

parsed_case_code = filtered_df["case_code"].str.replace('-', '_')
filtered_df['bag_id'] = "RE_" + parsed_case_code + "_1_" + filtered_df["bersaglio"].astype(str)

In [13]:
# Filter to keep only the analyzed cases

features_dir = Path("/home/ubuntu/giodir/digitalPathology/data/features/uni_features_RE_all")

available_bags = {path.stem for path in features_dir.glob("*.npz")}
print("Analyzed bags:", len(available_bags))

avail_df = filtered_df[filtered_df['bag_id'].isin(available_bags)]

print(f"Total cases: {len(filtered_df)}, Available cases: {len(avail_df)},")

Analyzed bags: 550
Total cases: 538, Available cases: 538,


## LABEL TYPE

In [14]:
## binary like 0 vs 1+
avail_df["binary_difference"] = (avail_df["difference"] > 0).astype(int)

# keep only the important differences (set 0 where the difference is 0, 1 if it is 2 or more and None if it is 1)
avail_df["important_difference"] = avail_df["difference"].apply(lambda x: 0 if x == 0 else (1 if x >= 2 else None))

In [15]:
# Save in csv the three files like (bag_id, label_col)

basedir = Path("/home/ubuntu/giodir/digitalPathology/data/labels/all_cad_discordance_labels")
basedir.mkdir(exist_ok=True)


# binary diff
avail_df[["bag_id", "binary_difference"]].rename(columns={"binary_difference": "label"}).to_csv(
    basedir / "binary_diff_labels.csv", index=False)

# binary important diff
avail_df[["bag_id", "important_difference"]].rename(columns={"important_difference": "label"}).to_csv(
    basedir / "binary_important_diff_labels.csv", index=False)

# original diff
avail_df[["bag_id", "difference"]].rename(columns={"difference": "label"}).to_csv(
    basedir / "difference_labels.csv", index=False)
